In [1]:
import json
from pathlib import Path

In [2]:
data_path = Path("../data/raw/diseases.json")
output_path = Path("../data/processed/patient_dataset.json")

# Load the diseases
with open(data_path, "r", encoding="utf-8") as f:
    diseases = json.load(f)

print(f"Chargé avec succès {len(diseases)} profils de maladies pour alimenter la simulation.")
print("Exemple de maladie :", diseases[0]["name"])

Chargé avec succès 3 profils de maladies pour alimenter la simulation.
Exemple de maladie : Lupus érythémateux systémique


In [3]:
# We create the prompt to guide the LLM in its role play

DEFAULT_SYSTEM_PROMPT = """Tu es un patient virtuel dans un jeu d'enquête médicale. 
Tu consultes un médecin (le joueur). 
Voici ta pathologie secrète et tes symptômes :
- Maladie : {disease_name}
- Symptômes visibles : {visible_symptoms}
- Symptômes cachés : {hidden_symptoms}

Règles :
1. Ne révèle jamais le nom de ta maladie directement.
2. Ne donne tes symptômes cachés que si le joueur te pose une question précise ou insiste.
3. Adopte un ton naturel, légèrement inquiet et humain.
"""

In [ ]:
def generate_synthetic_dialogue(disease_info, system_prompt = DEFAULT_SYSTEM_PROMPT ) :
    """Generate an interactive dialogue scenario for a specific disease using the system prompt."""
    
    # We dynamically injecte disease data into the System Prompt

    formatted_system_prompt = system_prompt.format(
        disease_name=disease_info.get("name", "Inconnue"),
        visible_symptoms=", ".join(disease_info.get("visible_symptoms", [])),
        hidden_symptoms=", ".join(disease_info.get("hidden_symptoms", []))
    )
    
    # We structure the messages using a strict "user" / "assistant" format to ensure a simple chat_template structure.
    # The context is injected along with the doctor's question into the "user" role.

    context_and_doctor_content = (
        f"[CONTEXTE SYSTÈME]\n{formatted_system_prompt}\n\n"
        f"[MÉDECIN] Bonjour, qu'est-ce qui vous amène aujourd'hui ?"
    )

    patient_content = f"Bonjour docteur... Je ne me sens pas bien, j'ai notamment {', '.join(disease_info.get('visible_symptoms', ['des douleurs']))}."
    dialogue_turns = [
        {"role": "user", "content": context_and_doctor_content},
        {"role": "assistant", "content": patient_content}
    ]

    return dialogue_turns

In [8]:
dataset = []
for disease in diseases:
    scenario = generate_synthetic_dialogue(disease)
    dataset.append({
        "disease": disease["name"],
        "messages": scenario
    })

print(f"{len(dataset)} scénarios de dialogue générés.")

3 scénarios de dialogue générés.


In [9]:
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(dataset, f, ensure_ascii=False, indent=4)

print(f"Dataset sauvegardé avec succès dans : {output_path}")

Dataset sauvegardé avec succès dans : ..\data\processed\patient_dataset.json
